# Klasifikasi Naive Bayes Manual (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini mengimplementasikan **Gaussian Naive Bayes** dari nol, tanpa
scikit-learn. Model dan metriknya dihitung sendiri. Datanya dibaca dari berkas bersih
hasil notebook `02_bersihkan_data`, jadi tidak ada library machine learning yang dipakai.

Caranya: hitung mean dan std tiap fitur per kelas, lalu hitung probabilitas tiap kelas
dengan rumus Gaussian PDF dan pilih yang tertinggi.

Dataset yang dipakai: Stroke Prediction versi bersih dari notebook `02_bersihkan_data`
(5.109 baris). Hanya kolom numerik yang dipakai: age, hypertension, heart_disease,
avg_glucose_level, bmi. Target: stroke.

Pasangan notebook ini untuk KNN ada di `03a_knn_manual.ipynb`.
Versi scikit-learn menyusul di `04_model_library.ipynb`.


## 1. Muat Dataset

Data diambil langsung dari **berkas dataset bersih** yang dihasilkan notebook
`02_bersihkan_data`, yaitu `artifacts/stroke_bersih.csv` (5.109 baris, `bmi` sudah diisi
prediksi Ridge, `gender` "Other" sudah dibuang).

Karena pembersihannya sudah selesai di notebook `02`, notebook ini bisa fokus penuh ke
algoritmanya. Kolom `gender` tetap dimuat karena dibutuhkan skor medis di bagian 5.

In [ ]:
import pandas as pd
import numpy as np
import math
import random

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

## 2. Train-Test Split Manual

Split 80:20 tanpa library. Acak indeks dulu, lalu potong. Karena datanya tidak
seimbang (hanya 4,87% stroke), proporsi kelas di train dan test dicetak supaya
kelihatan apakah kedua bagian kebagian kasus stroke.

In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

## 3. Fungsi Evaluasi Manual

Semua metrik dihitung dari confusion matrix tanpa library.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 4. Gaussian Naive Bayes Manual

Hitung mean dan std tiap fitur per kelas, lalu hitung probabilitas dengan rumus
Gaussian PDF. Pilih kelas dengan probabilitas tertinggi.

Perhitungan dikerjakan dalam bentuk log supaya angka sangat kecil tidak langsung
menjadi nol karena perkalian berulang.

In [ ]:
def gaussian_pdf(x, mean, std):
    """Hitung probabilitas Gaussian."""
    if std == 0:
        return 1.0 if x == mean else 0.0001
    eksponen = math.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (math.sqrt(2 * math.pi) * std)) * eksponen


class NaiveBayesManual:
    """Gaussian Naive Bayes buatan sendiri."""

    def fit(self, X, y):
        n, m = X.shape
        self.kelas = sorted(set(y))
        self.prior = {}
        self.mean = {}
        self.std = {}

        for c in self.kelas:
            Xc = X[y == c]
            self.prior[c] = len(Xc) / n
            self.mean[c] = Xc.mean(axis=0)
            self.std[c] = Xc.std(axis=0)

    def predict(self, X):
        y_pred = []
        for titik in X:
            prob_kelas = {}
            for c in self.kelas:
                prob = math.log(self.prior[c])
                for i in range(len(titik)):
                    prob += math.log(gaussian_pdf(titik[i], self.mean[c][i], self.std[c][i]) + 1e-10)
                prob_kelas[c] = prob
            y_pred.append(max(prob_kelas, key=prob_kelas.get))
        return np.array(y_pred)


nb = NaiveBayesManual()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Naive Bayes selesai.")

In [ ]:
metrik_nb = hitung_metrik(y_test, y_pred_nb)

print("=" * 50)
print("HASIL NAIVE BAYES")
print("=" * 50)
print(f"  Akurasi:  {metrik_nb['akurasi']}")
print(f"  Presisi:  {metrik_nb['presisi']}")
print(f"  Recall:   {metrik_nb['recall']}")
print(f"  F1-score: {metrik_nb['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_nb['tp']}  FP={metrik_nb['fp']}")
print(f"  FN={metrik_nb['fn']}  TN={metrik_nb['tn']}")

### 4a. Lihat Parameternya

Naive Bayes mudah diperiksa karena parameternya cuma mean dan std tiap fitur per
kelas. Berguna untuk memastikan sesuai akal sehat.

In [ ]:
nama_fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

for c in nb.kelas:
    label = "stroke" if c == 1 else "tidak stroke"
    print(f"Kelas {c} ({label}), prior {nb.prior[c]:.4f}")
    for i, nama in enumerate(nama_fitur):
        print(f"  {nama:18s} mean={nb.mean[c][i]:8.2f}  std={nb.std[c][i]:8.2f}")
    print()

## 5. Pembanding: Skor Risiko Stroke Klinis (CHA2DS2-VASc)

Sebelum ada machine learning, dunia medis sudah punya cara menghitung risiko stroke.
Salah satu yang paling dikenal adalah **CHA2DS2-VASc**, skor yang dipakai untuk
memperkirakan risiko stroke pada pasien fibrilasi atrium.

Skornya hanyalah penjumlahan poin:

$$\text{Skor} = C + H + A_2 + D + S_2 + V + A + Sc$$

dengan tiap komponen bernilai:

| Lambang | Arti | Poin |
|---|---|---|
| $C$ | Congestive heart failure (gagal jantung) | 1 |
| $H$ | Hypertension (hipertensi) | 1 |
| $A_2$ | Age $\geq$ 75 tahun | 2 |
| $D$ | Diabetes mellitus | 1 |
| $S_2$ | Stroke atau TIA sebelumnya | 2 |
| $V$ | Vascular disease (penyakit vaskular) | 1 |
| $A$ | Age 65 sampai 74 tahun | 1 |
| $Sc$ | Sex category (perempuan) | 1 |

### Adaptasi ke dataset kami

Dua komponen tidak bisa dipakai, dan ini harus disebut apa adanya:

- $S_2$ (riwayat stroke) justru **label yang mau diprediksi**, jadi tidak boleh jadi fitur.
- $V$ (penyakit vaskular) tidak ada di dataset.

Sisanya dipakai apa adanya, dengan dua proksi yang juga harus disebut:

- $C$ memakai kolom `heart_disease`
- $D$ memakai `avg_glucose_level >= 126` mg/dL, ambang diabetes menurut ADA

Karena itu skor maksimum di sini **6**, bukan 9.

Skor ini tidak dilatih dan tidak belajar dari data, ia aturan tetap. Justru itu gunanya
di sini: jadi pembanding "cara dokter" terhadap "cara machine learning".

In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

## 6. Kesimpulan

Gaussian Naive Bayes manual berhasil dijalankan dan dievaluasi, sepenuhnya tanpa
scikit-learn.

| Metrik | Nilai |
|---|---|
| Akurasi | 0,8727 |
| Presisi | 0,2149 |
| Recall | 0,4262 |
| F1 | 0,2857 |

Confusion matrix: TP=26, FP=95, FN=35, TN=865. Dari 61 pasien stroke di data uji, 26
berhasil terdeteksi.

### Dibandingkan aturan medis

Skor CHA2DS2-VASc di bagian 5 tidak dilatih sama sekali, tetapi hasilnya sepadan:

| Cara | Recall | Presisi | F1 |
|---|---|---|---|
| Skor medis, ambang skor >= 2 | 0,6885 | 0,1368 | 0,2283 |
| Skor medis, ambang skor >= 3 | 0,5082 | 0,2109 | 0,2981 |
| Naive Bayes | 0,4262 | 0,2149 | 0,2857 |

Naive Bayes hanya unggul tipis pada presisi (0,2149 lawan 0,2109). Pada recall dan F1 ia
justru di bawah ambang skor >= 3 (0,4262 lawan 0,5082, dan 0,2857 lawan 0,2981).
Kesimpulannya: untuk data sesederhana ini, **aturan medis yang tidak belajar apa pun
sepadan, bahkan sedikit lebih baik, dibanding Naive Bayes**. Itu alasan kuat untuk selalu
membandingkan model dengan cara sederhana sebelum mengklaim berhasil.

Catatan penting:

1. **Asumsi independensi antar fitur.** Naive Bayes menganggap tiap fitur saling bebas
   setelah diketahui kelasnya. Di data ini asumsi itu tidak sepenuhnya benar (usia dan
   hipertensi jelas berkorelasi), tetapi model tetap layak dipakai sebagai pembanding.
2. **Parameternya masuk akal secara medis.** Rata-rata usia kelas stroke 68,3 tahun
   lawan 41,8 tahun pada kelas tidak stroke, dan glukosanya 132,5 lawan 104,5. Model
   menangkap sinyal yang sejalan dengan pengetahuan medis.
3. **Tidak terpengaruh skala fitur** karena memakai mean dan std per kelas. Berbeda
   dari KNN di `03a_knn_manual.ipynb` yang wajib di-scaling.
4. **Recall masih jauh dari ideal.** Data hanya punya 4,87% kasus stroke, jadi model
   tetap cenderung menebak kelas mayoritas dan akurasi tetap menyesatkan.

Langkah berikutnya: versi scikit-learn di `04_model_library.ipynb`.